# ESM C (Cambrian)

![ESM C (Cambrian)](https://proto-bio.github.io/proto-assets/images/tool/esmc/hero.png)

ESM C is an embedding-focused masked protein language model from Biohub. Unlike ESM3, ESM C is purpose-built for producing high-quality contextualized embeddings: there is no sample/score interface.

Three open-weights variants are exposed here, all MIT-licensed: `esmc_300m` (960-dim), `esmc_600m` (1152-dim), and `esmc_6b` (2560-dim). This notebook uses the 300M default; `esmc_6b` pulls roughly 25 GB of weights on first use.

The toolkit also exposes sparse autoencoders over ESM C's activations, covered in the second half of this notebook.


In [1]:
from proto_tools.utils.notebook_docs import (
    display_overview, display_api_reference, display_doc_link, display_available_tools,
)
display_doc_link("esmc")
display_overview("esmc")

# ESM C (Cambrian)

ESM C ("Cambrian") is [Biohub](https://biohub.ai)'s embedding-focused protein language model. This toolkit wraps the `esmc_300m`, `esmc_600m`, and `esmc_6b` models to produce per-sequence embeddings and optional per-position scores (logits) from supplied protein sequences, and to decompose those activations with sparse autoencoders (SAEs) into a large, sparsely-active feature space that is easier to interpret than raw embeddings. It does not support sequence sampling or scoring.

## Available tools

In [2]:
display_available_tools("esmc")

- **`run_esmc_embeddings()`** — Extract protein sequence embeddings and logits using ESM C (Cambrian)
- **`run_esmc_sae_features()`** — Decompose ESM C activations into interpretable sparse autoencoder features

### `run_esmc_embeddings`

Returns a `SequenceEmbedding` per input sequence containing a mean-pooled embedding, the attention mask, and (optionally) per-position amino-acid logits.

In [3]:
import numpy as np

from proto_tools.tools.masked_models.esmc import (
    ESMCEmbeddingsConfig, ESMCEmbeddingsInput, run_esmc_embeddings,
)

# Two hemoglobin chains (alpha, beta) are close homologs; GFP is an unrelated control.
hba = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
hbb = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"
gfp = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"

result = run_esmc_embeddings(ESMCEmbeddingsInput(sequences=[hba, hbb, gfp]), ESMCEmbeddingsConfig())
embs = [np.array(r.mean_embedding) for r in result.results]

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Embedding dim: {embs[0].shape[0]}")
print(f"cosine(HBA, HBB) = {cosine(embs[0], embs[1]):.3f}  (homologous globins)")
print(f"cosine(HBA, GFP) = {cosine(embs[0], embs[2]):.3f}  (unrelated)")

Running run_esmc_embeddings [00:00]

Embedding dim: 960
cosine(HBA, HBB) = 0.971  (homologous globins)
cosine(HBA, GFP) = 0.059  (unrelated)


### `run_esmc_sae_features`

Sparse autoencoders decompose ESM C's internal activations into a large, sparsely-active feature space that is easier to interpret than raw embeddings. Each residue activates exactly `k` features out of a codebook of thousands.

**The config selects a model, it does not tune one.** Each combination of backbone, layer, `k`, and `codebook_size` is a separately trained SAE with its own learned dictionary, so `model_checkpoint`, `sae_target`, `layers`, `k`, and `codebook_size` together name which of Biohub's 97 published SAEs to load.

A consequence: within one SAE a feature index always denotes the same learned concept, so activations are comparable across proteins. Indices are *not* comparable between different SAEs — including different layers of the same backbone.

This notebook uses the 300M backbone at its default layer. See the [SAE overview card](https://huggingface.co/biohub/ESMC-SAE-Overview) for the full set of variants.


In [4]:
from proto_tools.tools.masked_models.esmc import (
    ESMCSAEFeaturesConfig, ESMCSAEFeaturesInput, run_esmc_sae_features,
)

# Human ubiquitin (P0CG48 residues 1-76) — small, well characterized.
ubiquitin = (
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
)
inputs = ESMCSAEFeaturesInput(sequences=[ubiquitin])
result = run_esmc_sae_features(inputs, ESMCSAEFeaturesConfig())

features = result.results[0].layers[0]
print(f"layer {features.layer}: {len(features.feature_indices)} residues, "
      f"{len(features.feature_indices[0])} active features each")

Running run_esmc_sae_features [00:00]

layer 23: 76 residues, 64 active features each


## Which features fire most consistently?

A feature active at many positions is tracking something global about the protein; one active at a single position is tracking a local motif.

In [5]:
from collections import Counter

counts = Counter(idx for row in features.feature_indices for idx in row)
for feature_index, n in counts.most_common(5):
    print(f"feature {feature_index:>6}: active at {n:>3}/{len(features.feature_indices)} residues")

feature   9211: active at  76/76 residues
feature    386: active at  76/76 residues
feature  13844: active at  76/76 residues
feature  15182: active at  76/76 residues
feature   5562: active at  76/76 residues


## Comparing layers

Attaching several SAEs in one call runs the backbone once. Only the requested layers are downloaded.

In [6]:
multi = run_esmc_sae_features(
    inputs, ESMCSAEFeaturesConfig(layers=[5, 15, 23])
)
for layer in multi.results[0].layers:
    strongest = max(
        (m[0], i[0]) for i, m in zip(layer.feature_indices, layer.feature_magnitudes)
    )
    print(f"layer {layer.layer:>2}: strongest activation {strongest[0]:.2f} on feature {strongest[1]}")

Running run_esmc_sae_features [00:00]

layer  5: strongest activation 8.32 on feature 5076
layer 15: strongest activation 13.09 on feature 13134
layer 23: strongest activation 13.53 on feature 12880


## What do the features mean?

Biohub generated natural-language descriptions for one SAE — `ESMC-6B-sae-layer60-k64-codebook16384` — by analyzing how each feature activates across UniRef90. `describe_sae_features` fetches them from the public ESM Atlas API. Feature indices are specific to the SAE that produced them, so these apply only to features from that exact SAE; the 6B defaults resolve to it.

**Rank by normalized activation, not raw magnitude.** The strongest raw activations are dominated by features that fire on almost every protein and carry little information. Each record ships the statistics needed to correct for that: `(activation / uniref90_max_activation) * uniref90_idf` scales a feature to [0, 1] and upweights rare, distinctive ones. On ubiquitin, raw magnitude surfaces an "Unknown generic feature"; normalized, the top hit is a ubiquitin-like domain detector.

In [7]:
from proto_tools.tools.masked_models.esmc import DESCRIBED_SAE_REPO, describe_sae_features

described_config = ESMCSAEFeaturesConfig(model_checkpoint="esmc_6b")
assert described_config.sae_repo == DESCRIBED_SAE_REPO

features_6b = run_esmc_sae_features(inputs, described_config).results[0].layers[0]

# Peak activation per feature across the sequence.
peak = {}
for indices, magnitudes in zip(features_6b.feature_indices, features_6b.feature_magnitudes):
    for feature_id, magnitude in zip(indices, magnitudes):
        peak[feature_id] = max(peak.get(feature_id, 0.0), magnitude)

# Over 1,200 features fire somewhere on this sequence. Describing the 50 strongest
# and re-ranking those gives the same answer as describing all of them.
candidates = sorted(peak, key=peak.get, reverse=True)[:50]
records = describe_sae_features(candidates)

def normalized(feature_id):
    """Scale to [0, 1] by the feature's UniRef90 maximum, then weight by its IDF."""
    record = records[feature_id]
    return (peak[feature_id] / record["uniref90_max_activation"]) * record["uniref90_idf"]

for feature_id in sorted(records, key=normalized, reverse=True)[:5]:
    record = records[feature_id]
    print(f"{normalized(feature_id):5.2f}  feature {feature_id:>5}  {record['label']}")
    print(f"       category={record['category']}")

Running run_esmc_sae_features [00:00]

 2.45  feature  3995  Ubiquitin-like domain detector
       category=Domain
 2.12  feature  7865  ERAD–p97 UBL/UBX proteostasis
       category=Domain
 1.75  feature  3230  Short amphipathic helical interfaces
       category=Interaction site
 1.64  feature  2681  Short beta strand-coil segment
       category=Compositional bias
 1.62  feature   602  Acidic strand-loop docking patches
       category=Interaction site


## Export

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)
result.export(name="esmc_sae_features", export_path=output_dir, file_format="csv")
print(sorted(p.name for p in output_dir.glob("esmc_sae_features*")))

['esmc_sae_features.csv']


## API reference

In [9]:
display_api_reference("esmc", "input", "run_esmc_embeddings")
display_api_reference("esmc", "config", "run_esmc_embeddings")
display_api_reference("esmc", "output", "run_esmc_embeddings")
display_api_reference("esmc", "input", "run_esmc_sae_features")
display_api_reference("esmc", "config", "run_esmc_sae_features")
display_api_reference("esmc", "output", "run_esmc_sae_features")


**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

**Config** — `ESMCEmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>batch_size</code> | <code>int</code> | <code>8</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>model_checkpoint</code> | <code>Literal['esmc_300m', 'esmc_600m', 'esmc_6b']</code> | <code>'esmc_300m'</code> | ESM C weights variant; larger checkpoints embed better but need more GPU memory |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |
| <code>repr_layer</code> | <code>int</code> | <code>-1</code> | Transformer layer index for embeddings; -1 returns post-norm output, others select pre-norm |

**Output** — `ESMCEmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[SequenceEmbedding]</code> | required | Per-sequence embedding results |

**`SequenceEmbedding`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>mean_embedding</code> | <code>list[float]</code> | required | Mean-pooled embedding vector (averaged over sequence length) |
| <code>attention_mask</code> | <code>list[int]</code> | required | Binary mask: 1 = valid position, 0 = padding |
| <code>logits</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Per-position amino acid logits (seq_len, vocab_size) |
| <code>projection</code> | <code>Projection2D &#124; None</code> | <code>None</code> | 2D UMAP projection of this sequence's embedding within the call's batch |

**`Projection2D`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>x</code> | <code>float</code> | required | First reduced coordinate |
| <code>y</code> | <code>float</code> | required | Second reduced coordinate |

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

**Config** — `ESMCSAEFeaturesConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>model_checkpoint</code> | <code>Literal['esmc_300m', 'esmc_600m', 'esmc_6b']</code> | <code>'esmc_300m'</code> | ESM C backbone whose activations the SAE decomposes |
| <code>layers</code> | <code>list[int] &#124; None</code> | <code>None</code> | Backbone layers to attach SAEs to; None uses the ~75%-depth sweep layer |
| <code>sae_target</code> | <code>Literal['hidden_states', 'mlp_outputs']</code> | <code>'hidden_states'</code> | Selects the SAE trained on this activation source: residual stream or per-layer MLP |
| <code>k</code> | <code>Literal[16, 32, 64, 128, 256, 512]</code> | <code>64</code> | Selects the SAE trained with this many active features per residue; 64 serves any layer |
| <code>codebook_size</code> | <code>Literal[8192, 16384, 32768, 65536, 131072]</code> | <code>16384</code> | Selects the SAE trained with this many features in total; larger splits concepts finer |
| <code>backbone</code> | <code>Literal['transformers', 'esm']</code> | <code>'transformers'</code> | Which ESM C implementation supplies activations; 'esm' reuses the esmc toolkit weights |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |

**Output** — `ESMCSAEFeaturesOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[SequenceSAEFeatures]</code> | required | Per-sequence sparse SAE features |

**`SequenceSAEFeatures`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | Input sequence these features were computed from |
| <code>layers</code> | <code>list[SAELayerFeatures]</code> | required | Per-layer sparse features for this sequence, ascending by layer |

**`SAELayerFeatures`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>layer</code> | <code>int</code> | required | Backbone transformer layer the SAE reads from |
| <code>feature_indices</code> | <code>list[list[int]]</code> | required | Active codebook indices per residue, descending by magnitude |
| <code>feature_magnitudes</code> | <code>list[list[float]]</code> | required | Activation value of each active feature, descending |